# 01 — Ontology basics

The Digicities platform is built around a shared RDF/OWL ontology. Every digital twin you build is a set of **instances** of classes defined in that ontology, connected by its object properties.

This notebook gets you familiar with:

1. Connecting to the triplestore via the Python backend
2. Inspecting the core ontology (classes, properties, units)
3. Querying the Alpine Village sample data
4. Using namespaces / prefixes properly

Prerequisite: complete [`00_setup.md`](00_setup.md).

## 1.1 Connect to the triplestore

The backend client is **backend-agnostic**: it talks to whatever `TRIPLESTORE_BACKEND` selects — Apache Jena Fuseki by default, Ontotext GraphDB if you opt into the overlay — at the URL in `GRAPHDB_URL`. With `AUTH_DISABLED=true`, pass `token="local"` and it skips the bearer `Authorization` header. (Fuseki still needs HTTP Basic admin auth for *writes*; that comes from the `FUSEKI_ADMIN_*` env vars set in the cell above.)

In [ ]:
import os, sys, pathlib
# Make the repo root importable without requiring `pip install -e .`
sys.path.insert(0, str(pathlib.Path().resolve().parent))
# Default stack ships Apache Jena Fuseki on :3030. (The optional GraphDB
# overlay would be TRIPLESTORE_BACKEND=graphdb + GRAPHDB_URL=http://localhost:7201.)
os.environ.setdefault("TRIPLESTORE_BACKEND", "fuseki")
os.environ.setdefault("GRAPHDB_URL", "http://localhost:3030")
# Fuseki needs admin auth for writes (the upload_ttl / sparql_update cells below).
# These are the docker-compose defaults — override if you changed FUSEKI_ADMIN_PASSWORD.
os.environ.setdefault("FUSEKI_ADMIN_USER", "admin")
os.environ.setdefault("FUSEKI_ADMIN_PASSWORD", "admin")

from backend.graphdb import GraphDBClient

client = GraphDBClient(token="local", selected_repo="workspace_demo")
assert client.test_connection(), "Triplestore did not respond — is `docker compose up` running?"
client.auth_mode, client.base_url, client.repository

## 1.2 Load the Alpine Village sample

We load the TTL into a **named graph** so it stays isolated from other data in the repo. Re-running this cell is safe — `replace_existing=True` wipes the graph first.

In [ ]:
SAMPLE_GRAPH = "<https://digicities.info/tutorial/alpine_village>"

with open("sample_data/alpine_village.ttl", "r", encoding="utf-8") as f:
    client.upload_ttl(
        ttl_str=f.read(),
        graph_name=SAMPLE_GRAPH,
        replace_existing=True,
    )

client.sparql_api_query(f"""
    SELECT (COUNT(*) AS ?triples) WHERE {{ GRAPH {SAMPLE_GRAPH} {{ ?s ?p ?o }} }}
""", out_format="df")

## 1.3 Inspect the core ontology

The core ontology defines the vocabulary you model with. In this tutorial dataset it lives in the **default graph** — a query with no `GRAPH { … }` clause reads it, while your instance data lives in named graphs. So a clause-less query hits the ontology; a `GRAPH <…>` query hits your data.

Key top-level classes for energy modelling:

In [ ]:
client.sparql_api_query("""
    PREFIX dici_onto: <https://digicities.info/ontology#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX owl:  <http://www.w3.org/2002/07/owl#>

    SELECT ?label ?class WHERE {
      ?class a owl:Class ; rdfs:label ?label .
      FILTER(?class IN (
        dici_onto:EnergyConsumer,
        dici_onto:EnergyGenerator,
        dici_onto:EnergyStorage,
        dici_onto:EnergyConverter,
        dici_onto:Network,
        dici_onto:Flow,
        dici_onto:EnergyCarrier
      ))
    } ORDER BY ?label
""", out_format="df")

Attributes are first-class resources too. Every piece of data hanging off a component — a rated power, a cost, a demand profile — is an instance of an `Attribute` subclass. The main flavours:

| Class | When to use |
|---|---|
| `PhysicalAttribute` | A scalar with a QUDT unit (e.g. `8.0 KiloW`) |
| `DynamicAttribute` | Same, but backed by a time series |
| `SimpleCostAttribute` | A single price (e.g. `0.25 CHF/kWh`) |
| `UnitBasedCostAttribute` | A price per unit (e.g. `1600 CHF/kW` installed capex) |
| `CurveAttribute` | An efficiency / characteristic curve as (x, y) points |
| `CategoricalAttribute` | A string value from a controlled vocabulary |

## 1.4 Explore Alpine Village

How many components of each top-level type are in the village?

In [ ]:
client.sparql_api_query(f"""
    PREFIX dici_onto: <https://digicities.info/ontology#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?type (COUNT(?c) AS ?n) WHERE {{
      GRAPH {SAMPLE_GRAPH} {{
        ?c a ?type .
        FILTER(STRSTARTS(STR(?type), 'https://digicities.info/ontology#'))
      }}
    }} GROUP BY ?type ORDER BY DESC(?n)
""", out_format="df")

Let's pull every component and its attributes together. This is the pattern you'll reuse everywhere — a component plus a list of its (attribute-name, value, unit) triples.

In [ ]:
df = client.sparql_api_query(f"""
    PREFIX dici_onto: <https://digicities.info/ontology#>
    PREFIX qudt:      <http://qudt.org/schema/qudt/>
    PREFIX rdfs:      <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?component ?comp_label ?attr_label ?value ?unit WHERE {{
      GRAPH {SAMPLE_GRAPH} {{
        ?component rdfs:label ?comp_label ;
                   dici_onto:hasAttribute ?attr .
        ?attr rdfs:label ?attr_label ;
              qudt:value ?value .
        OPTIONAL {{ ?attr dici_onto:hasUnitLabel ?unit }}
      }}
    }} ORDER BY ?comp_label ?attr_label
""", out_format="df")
df

## 1.5 Follow the flows

`Flow` resources wire the components together. Each flow has a carrier (electricity, heat, etc.) and connects a source component to a destination component.

In [ ]:
client.sparql_api_query(f"""
    PREFIX dici_onto: <https://digicities.info/ontology#>
    PREFIX rdfs:      <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?flow_label ?carrier_label WHERE {{
      GRAPH {SAMPLE_GRAPH} {{
        ?flow a/rdfs:subClassOf* dici_onto:Flow ;
              rdfs:label ?flow_label ;
              dici_onto:carriesEnergyCarrier ?carrier .
        ?carrier rdfs:label ?carrier_label .
      }}
    }} ORDER BY ?flow_label
""", out_format="df")

## 1.6 A note on namespaces

Every URI in the ontology uses one of a handful of prefixes. Stick to them so your data joins up with the core schema:

| Prefix | Namespace | For |
|---|---|---|
| `dici_onto:` | `https://digicities.info/ontology#` | classes + properties from the core ontology |
| `qudt:` | `http://qudt.org/schema/qudt/` | quantity-kind / unit machinery |
| `unit:` | `http://qudt.org/vocab/unit/` | the actual unit IRIs (`unit:KiloW`, `unit:M2`, …) |
| `rdfs:` | `http://www.w3.org/2000/01/rdf-schema#` | labels and subclassing |
| `xsd:` | `http://www.w3.org/2001/XMLSchema#` | literal datatypes |

Your own instances go under a project namespace that is **not** `dici_onto:` — the Alpine Village uses `https://digicities.info/tutorial/alpine_village/`. The rule of thumb: the core ontology is read-only; your instances live in workspace-specific namespaces.

Not every triple store keeps a server-side prefix registry, but you can always see which vocabularies your data actually *uses* by reducing each predicate to its namespace — a portable SPARQL query that works on any backend:

In [ ]:
client.sparql_api_query(f"""
    SELECT ?namespace (COUNT(*) AS ?uses) WHERE {{
      GRAPH {SAMPLE_GRAPH} {{ ?s ?p ?o }}
      # Strip the local name (everything after the last # or /) to get the namespace.
      BIND(REPLACE(STR(?p), "[^#/]*$", "") AS ?namespace)
    }} GROUP BY ?namespace ORDER BY DESC(?uses)
""", out_format="df")

## Next

Notebook 02 builds on this by constructing a new component (a gas boiler to retrofit Building C) programmatically, generating TTL, and uploading it to the same named graph.